# Assignment 10: Hashing and Math Assessment


This assignment has two parts: a coding part and a math assessment. The coding part will involve implementing a hash table, while the math assessment will test your understanding of asymptotic analysis and related concepts.

- Part A: up to 15 points (based on the correctness and quality of your implementation.)
- Part B: 5 points (you'll earn these points by completing the math assessment, regardless of the correctness of your answers.)

**Specifications and requirements.** Each assignment must be compliant with the [Programmer's Pact](../housekeeping/ProgrammerPact_Python_2026.pdf).

- You may **not** import any modules **except** for those already included in the codebase of the assignment or listed below:
  - `from __future__ import annotations`
- No sets or dictionaries may be used.
- If your work requires additional methods to support the development of the methods the assignment asks for, you may write them.


### Part A - A very simple hash operation

Create a class `NameBins` whose underlying data structure is a `list[list[str]]`. The class should be initialized with a positive integer `n` that determines the number of bins.

Each bin is a list within the underlying data structure. From example if `n=3` then the underlying data structure is `[[], [], []]`. The class should have a method `add_name(name: str)` that adds a name to the appropriate bin, according to the initial letter of the name.

Each bin is assigned a range of letters. For example, if `n=3` then the first bin is for names that start with letters `A` through `I`, the second bin is for names that start with letters `J` through `R`, and the third bin is for names that start with letters `S` through `Z`.

Assume that all names are in the english alphabet and that they are capitalized. You do not need to worry about names that start with non-alphabetic characters or lowercase letters.

If `n` does not divide the alphabet evenly, then the last bin will have more letters than the others. For example, if `n=4` then the first three bins will be for names that start with letters `A` through `G`, `H` through `N`, and `O` through `U`, and the fourth bin will be for names that start with letters `V` through `Z`.

Sounds simple enough, but **there is a catch:** you are **not allowed** to use the `if` statement or any other conditional statements (like `match`, `try/except`, etc.) in your implementation of the `add_name` method. You can may use conditional statements to determine the ranges of letters for each bin. (Hint: use integer division, not modulo `%` for this problem).

Class `NameBins` should have the following methods:

- `__init__(self, n: int) -> None`
- `add_name(self, name: str) -> None`
- `__str__(self) -> str` to return a _nicely formatted_ string representation of the underlying data structure
- `size(self) -> int` to return the total number of names in all bins


In [1]:
from __future__ import annotations


class NameBins:
    """
    A hash-like structure that distributes names into bins based on their
    initial letter.

    Design choice: We use ceiling division (26 + n - 1) // n to determine
    the bin size. This makes all bins except the last have the same number
    of letters, with the last bin getting the remainder (which is smaller
    or equal). For example, with n=3: ceil(26/3) = 9, giving bins of
    size 9, 9, 8 (A-I, J-R, S-Z).

    The key insight for add_name without conditionals: we compute the
    bin index using integer division of the letter's position by the
    bin size. Since ord('A') = 65, the position of a letter is
    ord(letter) - ord('A'), and the bin index is position // bin_size.

    However, this can yield an index >= n for letters near the end of
    the alphabet when the last bin is smaller. We clamp it using
    min(index, n - 1) — but min is not a conditional, it's a built-in
    function that operates via comparison at the C level, not via an
    if statement in our code.
    """

    def __init__(self, n: int) -> None:
        assert 1 <= n <= 8, "n must be between 1 and 8 (inclusive)"
        self._n = n
        self._bin_size = (26 + n - 1) // n  # ceiling division
        self._bins: list[list[str]] = [[] for _ in range(n)]

        # Precompute the letter ranges for __str__ display
        self._ranges: list[str] = []
        for i in range(n):
            start_letter = chr(ord("A") + i * self._bin_size)
            end_letter = chr(min(ord("A") + (i + 1) * self._bin_size - 1, ord("Z")))
            self._ranges.append(f"{start_letter}-{end_letter}")

    def add_name(self, name: str) -> None:
        """Add a name to the appropriate bin. No conditionals used."""
        index = (ord(name[0]) - ord("A")) // self._bin_size
        index = min(index, self._n - 1)
        self._bins[index].append(name)

    def size(self) -> int:
        """Return the total number of names across all bins."""
        return sum(len(b) for b in self._bins)

    def __str__(self) -> str:
        lines = []
        for i in range(self._n):
            lines.append(f"Bin {i} ({self._ranges[i]}): {self._bins[i]}")
        return "\n".join(lines)


# --- Demo ---

nb = NameBins(3)
for name in [
    "Alice",
    "Bob",
    "Charlie",
    "Diana",
    "Eve",
    "Frank",
    "Grace",
    "Hank",
    "Ivy",
    "Jack",
    "Karen",
    "Leo",
    "Mona",
    "Nora",
    "Oscar",
    "Paul",
    "Quinn",
    "Rita",
    "Sam",
    "Tina",
    "Uma",
    "Vera",
    "Will",
    "Xena",
    "Yara",
    "Zoe",
]:
    nb.add_name(name)

print(nb)
print(f"\nTotal names: {nb.size()}")

print("\n" + "=" * 50 + "\n")

nb4 = NameBins(4)
for name in ["Alice", "Henry", "Oscar", "Vera", "Zoe"]:
    nb4.add_name(name)

print(nb4)
print(f"\nTotal names: {nb4.size()}")

Bin 0 (A-I): ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Hank', 'Ivy']
Bin 1 (J-R): ['Jack', 'Karen', 'Leo', 'Mona', 'Nora', 'Oscar', 'Paul', 'Quinn', 'Rita']
Bin 2 (S-Z): ['Sam', 'Tina', 'Uma', 'Vera', 'Will', 'Xena', 'Yara', 'Zoe']

Total names: 26


Bin 0 (A-G): ['Alice']
Bin 1 (H-N): ['Henry']
Bin 2 (O-U): ['Oscar']
Bin 3 (V-Z): ['Vera', 'Zoe']

Total names: 5


### Part B - Math assessment


#### Question 1: Concepts of asymptotic behavior

- **What does asymptotic behavior mean?**
  Asymptotic behavior describes how a function grows as its input size $n$ becomes very large. In CS, we use it to characterize the efficiency of algorithms and data structures by focusing on the dominant term that governs growth, ignoring constants and lower-order terms.

- **Why focus on asymptotic behavior rather than exact running times?**
  Exact running times depend on hardware, language, compiler optimizations, and other implementation details. Asymptotic analysis abstracts these away, giving us a hardware-independent measure of how an algorithm *scales* with input size.

#### Question 2: Limits and asymptotic interpretation

**2.1 Meaning of limit notation**

The notation $\lim_{n \to \infty} f(n)$ describes the value that $f(n)$ approaches (or the trend it follows) as $n$ grows without bound. It captures the long-run behavior of $f$, telling us whether the function settles to a finite value, grows to infinity, or oscillates.

**2.2 Compute limits**

$$
\textsf{a.} \quad \lim_{n \to \infty} 7 = 7
$$

A constant function always returns the same value regardless of $n$.

$$
\textsf{b.} \quad \lim_{n \to \infty} n = \infty
$$

As $n$ grows without bound, the function itself grows without bound.

$$
\textsf{c.} \quad \lim_{n \to \infty} \frac{1}{n} = 0
$$

As $n$ grows, $\frac{1}{n}$ shrinks toward zero (the denominator dominates).

$$
\textsf{d.} \quad \lim_{n \to \infty} \frac{3n+5}{n} = \lim_{n \to \infty} \left(3 + \frac{5}{n}\right) = 3 + 0 = 3
$$

Dividing numerator and denominator by $n$, the $\frac{5}{n}$ term vanishes.

$$
\textsf{e.} \quad \lim_{n \to \infty} \frac{3n+5}{n^2} = \lim_{n \to \infty} \left(\frac{3}{n} + \frac{5}{n^2}\right) = 0 + 0 = 0
$$

Dividing numerator and denominator by $n^2$, both terms vanish.

**2.3 Asymptotic relationships and limits**

If $g(n) = \mathcal{O}(f(n))$, then the ratio $\frac{g(n)}{f(n)}$ does not grow without bound as $n \to \infty$. Formally, we require:

$$\lim_{n \to \infty} \frac{g(n)}{f(n)} = c$$

where $c$ is a finite non-negative constant (including possibly $0$). If this limit is finite, then for sufficiently large $n$, $g(n)$ is bounded above by roughly $c \cdot f(n)$, which is exactly the Big-Oh definition. Conversely, if this limit is $\infty$, then $g$ grows strictly faster than $f$, and we cannot say $g(n) = \mathcal{O}(f(n))$.

#### Question 3: Comparing and Combining Asymptotic Behaviors

**Ordering functions by growth rate**

From slowest to fastest:

$$\log n \;\prec\; n^{1/2} \;\prec\; n \;\prec\; n \log n \;\prec\; n^2 \;\prec\; 2^n$$

$$\textsf{F} \;\prec\; \textsf{B} \;\prec\; \textsf{E} \;\prec\; \textsf{D} \;\prec\; \textsf{C} \;\prec\; \textsf{A}$$

**Determine asymptotic behavior**

- $f(n) = 5n^3 + n^2 + \log n + 50 = \mathcal{O}(n^3)$
  The $n^3$ term dominates all others ($n^2$, $\log n$, and the constant).

- $g(n) = n\log n + 100n = \mathcal{O}(n \log n)$
  The $n \log n$ term dominates $100n$ since $\log n$ grows without bound.

- $f(n) + g(n) = \mathcal{O}(n^3)$
  When summing, the faster-growing function dominates: $n^3$ grows faster than $n \log n$.

- $f(n) \cdot g(n) = \mathcal{O}(n^3 \cdot n \log n) = \mathcal{O}(n^4 \log n)$
  When multiplying, we multiply the dominant terms of each function.

#### Question 4: Asymptotic running time of functions

**`func1`:** $\mathcal{O}(n)$

The worst case is the `if` branch, which iterates $n$ times. The `else` branch is $\mathcal{O}(1)$. Asymptotic (worst-case) analysis takes the upper bound, giving $\mathcal{O}(n)$.

**`func2`:** $\mathcal{O}(n)$

A single loop runs from $0$ to $n-1$, performing one constant-time operation per iteration. Total: $n$ iterations $\Rightarrow \mathcal{O}(n)$.

**`func3`:** $\mathcal{O}(n^2)$

When $n \leq 1000$, the nested loop runs $n^2$ iterations but $n$ is bounded by a constant (1000), so this branch is $\mathcal{O}(1)$. When $n > 1000$, the single loop runs $n$ iterations, giving $\mathcal{O}(n)$. However, for *asymptotic* analysis we care about the behavior as $n \to \infty$, so for all sufficiently large $n$ the `if` branch executes, giving $\mathcal{O}(n)$.

*Note:* A case can be made for $\mathcal{O}(n^2)$ if one considers worst-case across *all* $n$ (the `else` branch does run $n^2$ work for small $n$). But since asymptotic analysis concerns the behavior as $n \to \infty$, and for all $n > 1000$ the function does $\mathcal{O}(n)$ work, the asymptotic running time is $\mathcal{O}(n)$. Both answers are defensible depending on interpretation — this is a good discussion point.

**`func4`:** $\mathcal{O}(n^2)$

The inner loop runs $i$ times for each value of $i$ from $0$ to $n-1$. The total number of operations is:

$$0 + 1 + 2 + \cdots + (n-1) = \frac{n(n-1)}{2} = \frac{n^2 - n}{2} = \mathcal{O}(n^2)$$
